In [ ]:
import pandas as pd
from io import BytesIO
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url="http://localhost:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin"
)



NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.

In [ ]:
response = s3.get_object(
    Bucket="bronze",
    Key="reed/ingestion_timestamp=1776414536/data.json"
)

df = pd.read_json(BytesIO(response["Body"].read()))



  source                     collected_at  total  \
0   reed 2026-04-17 08:28:56.332439+00:00    250   
1   reed 2026-04-17 08:28:56.332439+00:00    250   
2   reed 2026-04-17 08:28:56.332439+00:00    250   
3   reed 2026-04-17 08:28:56.332439+00:00    250   
4   reed 2026-04-17 08:28:56.332439+00:00    250   

                                                data  
0  {'jobId': 56717237, 'employerId': 691046, 'emp...  
1  {'jobId': 56657308, 'employerId': 691046, 'emp...  
2  {'jobId': 56717234, 'employerId': 691046, 'emp...  
3  {'jobId': 56774223, 'employerId': 691046, 'emp...  
4  {'jobId': 56717235, 'employerId': 691046, 'emp...  


In [19]:
df["data"]

0      {'jobId': 56717237, 'employerId': 691046, 'emp...
1      {'jobId': 56657308, 'employerId': 691046, 'emp...
2      {'jobId': 56717234, 'employerId': 691046, 'emp...
3      {'jobId': 56774223, 'employerId': 691046, 'emp...
4      {'jobId': 56717235, 'employerId': 691046, 'emp...
                             ...                        
245    {'jobId': 56761784, 'employerId': 659815, 'emp...
246    {'jobId': 56761786, 'employerId': 659815, 'emp...
247    {'jobId': 56761787, 'employerId': 659815, 'emp...
248    {'jobId': 56761788, 'employerId': 659815, 'emp...
249    {'jobId': 56761758, 'employerId': 659815, 'emp...
Name: data, Length: 250, dtype: object

In [26]:
df = pd.json_normalize(df["data"])

print(df.head())

      jobId  employerId employerName employerProfileId employerProfileName  \
0  56717237      691046        Tesco              None                None   
1  56657308      691046        Tesco              None                None   
2  56717234      691046        Tesco              None                None   
3  56774223      691046        Tesco              None                None   
4  56717235      691046        Tesco              None                None   

                                            jobTitle locationName  \
0  Customer Delivery Driver - Carlisle Warwick Rd...     Carlisle   
1           Customer Delivery Driver - Newbury Extra      Newbury   
2  Customer Delivery Driver - Carlisle Warwick Rd...     Carlisle   
3              Customer Delivery Driver - Yate Extra         Yate   
4  Customer Delivery Driver - Carlisle Warwick Rd...     Carlisle   

   minimumSalary  maximumSalary currency expirationDate        date  \
0            NaN            NaN     None     

In [30]:
cols = [
    "jobId",
    "employerName",
    "jobTitle",
    "jobDescription",
    "locationName",
    "minimumSalary",
    "maximumSalary",
    "date",
    "expirationDate",
    "jobUrl",
    "currency"
]
df = df[cols]

In [31]:
df.head()

,jobId,employerName,jobTitle,jobDescription,locationName,minimumSalary,maximumSalary,date,expirationDate,jobUrl,currency
0,56717237,Tesco,Customer Delivery Driver - Carlisle Warwick Rd...,About the role Availability Window Days From t...,Carlisle,NaN,NaN,30/03/2026,29/05/2026,https://www.reed.co.uk/jobs/customer-delivery-...,None
1,56657308,Tesco,Customer Delivery Driver - Newbury Extra,About the role Availability Window Days From t...,Newbury,NaN,NaN,16/03/2026,29/05/2026,https://www.reed.co.uk/jobs/customer-delivery-...,None
2,56717234,Tesco,Customer Delivery Driver - Carlisle Warwick Rd...,About the role Availability Window Days From t...,Carlisle,NaN,NaN,30/03/2026,29/05/2026,https://www.reed.co.uk/jobs/customer-delivery-...,None
3,56774223,Tesco,Customer Delivery Driver - Yate Extra,About the role Availability Window Days From t...,Yate,NaN,NaN,13/04/2026,29/05/2026,https://www.reed.co.uk/jobs/customer-delivery-...,None
4,56717235,Tesco,Customer Delivery Driver - Carlisle Warwick Rd...,About the role Availability Window Days From t...,Carlisle,NaN,NaN,30/03/2026,29/05/2026,https://www.reed.co.uk/jobs/customer-delivery-...,None


In [ ]:
import re
mapping = {
    "Software Engineer": "Software Engineer",
    "Software Developer": "Software Engineer",
    "Data Eng": "Data Engineer",
    "Data Scientist ": "Data Scientist"
}

df["jobTitle"] = df["jobTitle"].replace(mapping)
df["jobTitle"] = (
    df["jobTitle"]
    .fillna("Unknown")
    .str.encode("latin1", errors="ignore").str.decode("utf-8", errors="ignore")
    .str.strip()
    .str.replace(r"\s*[-–|]\s.*$", "", regex=True)
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
    .str.title()
)

# #############
def clean_text(col):
    return (
        col.str.encode("latin1", errors="ignore")
           .str.decode("utf-8", errors="ignore")
           .str.replace(r"<[^>]*>", " ", regex=True)     # remove HTML
           .str.replace(r"\s+", " ", regex=True)         # normalize spaces
           .str.replace(r"[^\w\s.,;:()\-']", " ", regex=True)
           .str.strip()
    )

df["jobDescription"] = clean_text(df["jobDescription"])


# #############
df["locationName"] = (
    df["locationName"]
    .fillna("Unknown")
    .str.encode("latin1", errors="ignore").str.decode("utf-8", errors="ignore")
    .str.strip()
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.title()
)

df[["city", "country"]] = df["locationName"].str.split(",", expand=True, n=1)
df["city"] = df["city"].str.strip()
df["country"] = df["country"].str.strip()
df["locationName"] = df["locationName"].replace("", "Unknown")

# ################## maximumSalary + minimumSalary
df["minimumSalary"] = pd.to_numeric(df["minimumSalary"], errors="coerce")
df.loc[df["minimumSalary"] < 0, "minimumSalary"] = None
q_low = df["minimumSalary"].quantile(0.01)
q_high = df["minimumSalary"].quantile(0.99)

df = df[(df["minimumSalary"].isna()) | 
        ((df["minimumSalary"] >= q_low) & (df["minimumSalary"] <= q_high))
        ]
import numpy as np

df.loc[df["minimumSalary"] < 0, "minimumSalary"] = np.nan

# #########################
df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")
df["date_str"] = df["date"].dt.strftime("%Y-%m-%d")

# df["year"] = df["date"].dt.year
# df["month"] = df["date"].dt.month
# df["day"] = df["date"].dt.day
# df["day_of_week"] = df["date"].dt.day_name()
# from datetime import datetime

# df["days_since_posted"] = (pd.Timestamp("today") - df["date"]).dt.days


df["currency"] = (
    df["currency"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
)

currency_map = {
    "£": "GBP",
    "$": "USD",
    "€": "EUR"
}


df["currency"] = df["currency"].replace(["NONE", "NAN", "UNKNOWN"], "UNKNOWN")


df["currency"] = (
    df["currency"]
    .fillna("UNKNOWN")
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({
        "£": "GBP",
        "$": "USD",
        "€": "EUR"
    })
)

| Column          | Purpose           |
| --------------- | ----------------- |
| `minimumSalary` | numeric           |
| `maximumSalary` | numeric           |
| `currency`      | standardized code |
